In [ ]:
from langchain_community.chat_models import ChatOllama
from langchain_community.document_loaders import PyPDFLoader, Docx2txtLoader

In [ ]:
llm = ChatOllama(model="llama3:latest")

In [ ]:
llm.invoke("Hello")

AIMessage(content="Hello! It's nice to meet you. Is there something I can help you with, or would you like to chat?", additional_kwargs={}, response_metadata={'model': 'llama3:latest', 'created_at': '2026-05-19T05:26:34.931459Z', 'message': {'role': 'assistant', 'content': ''}, 'done': True, 'done_reason': 'stop', 'total_duration': 1488583875, 'load_duration': 97342334, 'prompt_eval_count': 11, 'prompt_eval_duration': 62239791, 'eval_count': 26, 'eval_duration': 1314769749}, id='lc_run--019e3eb3-3897-71a0-a8dd-ba2dbb134bd5-0', tool_calls=[], invalid_tool_calls=[])

In [ ]:
loader_pdf = PyPDFLoader(file_path="ashish_Gour.pdf")

In [ ]:
pdfData = loader_pdf.load()

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter, MarkdownTextSplitter

In [ ]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1200,chunk_overlap=200)

In [ ]:
chunks = splitter.split_documents(pdfData)

In [ ]:
from langchain_ollama import OllamaEmbeddings

In [ ]:
embeddings = OllamaEmbeddings(model="nomic-embed-text")

In [ ]:
from langchain_community.vectorstores import FAISS

In [ ]:
vector_store = FAISS.from_documents(chunks, embeddings)

vector_store.save_local("vs")

In [ ]:
local_store = FAISS.load_local(

    "vs",

    embeddings,

    allow_dangerous_deserialization=True

)

In [ ]:
local_store.get_by_ids([""])


[]

In [ ]:
query = "What databases ashish has used?"

retriever = local_store.as_retriever(

    search_type="similarity",

    search_kwargs={"k": 3}

)


In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

In [ ]:
prompt = ChatPromptTemplate.from_template("""

You are a helpful assistant. Use only the context below to answer.

Context:

{context}

Question:

{query}

Answer in a clear and short way.

""")

def getContext(docs):
    return ' '.join([doc.page_content for doc in docs])

In [ ]:
chain_context = retriever | RunnableLambda(getContext)

In [ ]:
chain = ({"context": chain_context, "query": RunnablePassthrough()}
         | prompt
         | llm 
         | StrOutputParser()
    
)

In [ ]:
chain.invoke("What is the description of project Calling App?")

'The Calling App, also known as HVS Softphone (Anti Block Solution), allows users to make VoIP calls using their internet connection in SIP-blocked countries.'

In [ ]:
chunks[0]

Document(metadata={'producer': 'PyPDF', 'creator': 'Microsoft Word', 'creationdate': '2025-12-16T11:19:49+00:00', 'author': 'Vera Reichlin-Meldegg', 'moddate': '2025-12-16T11:19:49+00:00', 'source': 'ashish_Gour.pdf', 'total_pages': 5, 'page': 0, 'page_label': '1'}, page_content='Professional \nSummary \n iOS Developer with over 12 years of experience building high-quality mobile \napplications across iOS and cross-platform ecosystems. Proficient in React \nNative, Swift, SwiftUI, Objective-C, and the full iOS SDK, with hands-on \nexperience in native module development (Swift/Kotlin), TurboModules, and JSI \nfor performance-critical features. Delivered scalable solutions across diverse \ndomains including e-commerce, healthcare, travel, publishing, and real \nestate. \n \nStrong expertise in Clean Architecture, SOLID principles, and architectural \npatterns such as MVC, MVVM, VIPER, and MVVMC. Experienced with modern \nstate management (Redux Toolkit, React Query), offline-first archi

In [ ]:
chunks

[Document(metadata={'producer': 'PyPDF', 'creator': 'Microsoft Word', 'creationdate': '2025-12-16T11:19:49+00:00', 'author': 'Vera Reichlin-Meldegg', 'moddate': '2025-12-16T11:19:49+00:00', 'source': 'ashish_Gour.pdf', 'total_pages': 5, 'page': 0, 'page_label': '1'}, page_content='Professional \nSummary \n iOS Developer with over 12 years of experience building high-quality mobile \napplications across iOS and cross-platform ecosystems. Proficient in React \nNative, Swift, SwiftUI, Objective-C, and the full iOS SDK, with hands-on \nexperience in native module development (Swift/Kotlin), TurboModules, and JSI \nfor performance-critical features. Delivered scalable solutions across diverse \ndomains including e-commerce, healthcare, travel, publishing, and real \nestate. \n \nStrong expertise in Clean Architecture, SOLID principles, and architectural \npatterns such as MVC, MVVM, VIPER, and MVVMC. Experienced with modern \nstate management (Redux Toolkit, React Query), offline-first arch